<a href="https://colab.research.google.com/github/maks1005mt/machine-learning-course/blob/main/ML_Labs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install kagglehub -q

In [4]:
import kagglehub
path = kagglehub.dataset_download("hugomathien/soccer")
print("Путь к датасету:", path)

100%|██████████| 32.7M/32.7M [00:00<00:00, 131MB/s]

Extracting files...


Путь к датасету: /root/.cache/kagglehub/datasets/hugomathien/soccer/versions/10


In [7]:
import sqlite3, pandas as pd, os

db_path = os.path.join(path, 'database.sqlite')
conn = sqlite3.connect(db_path)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

matches = pd.read_sql("SELECT * FROM Match;", conn)
print("Размер:", matches.shape)
matches.head()

                name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes
Размер: (25979, 115)


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.20
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.80,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.60
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.35,3.25,2.65,2.50,3.20,2.50,2.30,3.20,2.75
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,7.50,1.45,3.75,6.50,1.50,3.75,5.50,1.44,3.75,6.50
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.73,4.50,3.40,1.65,4.50,3.50,1.65,4.75,3.30,1.67


In [12]:
# ========== 1. РАЗМЕР ДАТАСЕТА ==========
N, M = matches.shape
print(f"Число объектов N = {N}")
print(f"Число признаков M = {M}")

# ========== 2. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ И КЛАССЫ ==========
# Создаём целевую переменную: исход матча
def get_result(row):
    if row['home_team_goal'] > row['away_team_goal']:
        return 'H'   # победа хозяев
    elif row['home_team_goal'] < row['away_team_goal']:
        return 'A'   # победа гостей
    else:
        return 'D'   # ничья

matches['result'] = matches.apply(get_result, axis=1)

# Количество классов
n_classes = matches['result'].nunique()
print(f"\nКоличество классов: {n_classes}")
print(f"Классы: {sorted(matches['result'].unique())}")

# Распределение классов
print("\nРаспределение классов (абсолютное):")
print(matches['result'].value_counts())

print("\nРаспределение классов (в %):")
print((matches['result'].value_counts(normalize=True) * 100).round(2))

# ========== 3. ПРИЗНАКИ ==========
print("\n" + "="*50)
print("ПРИЗНАКИ")
print("="*50)

# Все колонки
all_features = matches.columns.tolist()
print(f"\nВсего колонок: {len(all_features)}")

# Группы признаков
numerical_cols = matches.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = matches.select_dtypes(include=['object']).columns.tolist()

print(f"\nЧисловых признаков: {len(numerical_cols)}")
print(f"Категориальных признаков: {len(categorical_cols)}")
print(f"\nКатегориальные колонки: {categorical_cols}")

Число объектов N = 25979
Число признаков M = 116

Количество классов: 3
Классы: ['A', 'D', 'H']

Распределение классов (абсолютное):
result
H    11917
A     7466
D     6596
Name: count, dtype: int64

Распределение классов (в %):
result
H    45.87
A    28.74
D    25.39
Name: proportion, dtype: float64

ПРИЗНАКИ

Всего колонок: 116

Числовых признаков: 105
Категориальных признаков: 11

Категориальные колонки: ['season', 'date', 'goal', 'shoton', 'shotoff', 'foulcommit', 'card', 'cross', 'corner', 'possession', 'result']


В датасете 116 колонок. По типу pandas 105 числовых и 11 объектных. Но по смыслу объектные разделяются на: 1 настоящий категориальный (season), 1 дата (date), 1 целевая (result) и 8 XML-строк со статистикой матчей (goal, shoton, ...). То есть настоящих категориальных признаков — 1.

In [23]:
# Настоящие числовые = числовые, НО исключаем:
# - ID (6)
# - координаты игроков (44)

id_cols = [c for c in matches.columns if 'id' in c.lower()]
coord_cols = [c for c in matches.columns if '_X' in c or '_Y' in c]

numeric_all = matches.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_real = [c for c in numeric_all
                if c not in id_cols + coord_cols]

print(f"Числовых всего:              {len(numeric_all)}")
print(f"Из них ID:                   {len([c for c in numeric_all if c in id_cols])}")
print(f"Из них координат:            {len([c for c in numeric_all if c in coord_cols])}")
print(f"\nНАСТОЯЩИХ ЧИСЛОВЫХ ПРИЗНАКОВ: {len(numeric_real)}")

Числовых всего:              105
Из них ID:                   6
Из них координат:            44

НАСТОЯЩИХ ЧИСЛОВЫХ ПРИЗНАКОВ: 55


In [13]:
# ========== ПРОПУСКИ В ПРОЦЕНТАХ ==========
missing_count = matches.isnull().sum()
missing_pct = (missing_count / len(matches) * 100).round(2)

# Собираем в таблицу
missing_df = pd.DataFrame({
    'Пропусков': missing_count,
    'Процент': missing_pct
})

# Оставляем только колонки с пропусками и сортируем по убыванию
missing_df = missing_df[missing_df['Пропусков'] > 0].sort_values('Процент', ascending=False)

print(f"Всего колонок: {matches.shape[1]}")
print(f"Колонок с пропусками: {len(missing_df)}")
print(f"Колонок без пропусков: {matches.shape[1] - len(missing_df)}")
print(f"\nОбщий процент пропущенных значений в датасете: "
      f"{matches.isnull().sum().sum() / matches.size * 100:.2f}%")

print("\n" + "="*60)
print("ПРОПУСКИ ПО КОЛОНКАМ (топ-30):")
print("="*60)
print(missing_df.head(30).to_string())

Всего колонок: 116
Колонок с пропусками: 104
Колонок без пропусков: 12

Общий процент пропущенных значений в датасете: 13.52%

ПРОПУСКИ ПО КОЛОНКАМ (топ-30):
            Пропусков  Процент
PSA             14811    57.01
PSD             14811    57.01
PSH             14811    57.01
BSD             11818    45.49
GBA             11817    45.49
BSA             11818    45.49
BSH             11818    45.49
GBD             11817    45.49
GBH             11817    45.49
goal            11762    45.28
shotoff         11762    45.28
corner          11762    45.28
cross           11762    45.28
shoton          11762    45.28
card            11762    45.28
foulcommit      11762    45.28
possession      11762    45.28
SJD              8882    34.19
SJA              8882    34.19
SJH              8882    34.19
IWD              3459    13.31
IWA              3459    13.31
IWH              3459    13.31
LBH              3423    13.18
LBD              3423    13.18
LBA              3423    13.18
VCA  

In [14]:
# ========== ОБЩИЙ ПРОЦЕНТ ВЫБРОСОВ ==========
# Считаем по всем числовым колонкам сразу

def count_outliers_iqr(series):
    s = series.dropna()
    if len(s) == 0:
        return 0, 0
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((s < lower) | (s > upper)).sum()
    return outliers, len(s)

# Берём только числовые колонки
numeric = matches.select_dtypes(include=['int64', 'float64'])

total_values = 0
total_outliers = 0

for col in numeric.columns:
    n_out, n_total = count_outliers_iqr(numeric[col])
    total_outliers += n_out
    total_values += n_total

overall_pct = total_outliers / total_values * 100

print(f"Всего числовых значений: {total_values:,}")
print(f"Всего выбросов:         {total_outliers:,}")
print(f"ОБЩИЙ ПРОЦЕНТ ВЫБРОСОВ: {overall_pct:.2f}%")

Всего числовых значений: 2,414,496
Всего выбросов:         120,015
ОБЩИЙ ПРОЦЕНТ ВЫБРОСОВ: 4.97%


In [15]:
# ========== ВСЕ ТИПЫ ПРИЗНАКОВ ==========
print("ВСЕ ТИПЫ ДАННЫХ В ДАТАСЕТЕ (dtypes):")
print(matches.dtypes.value_counts())
print()

# ========== РАЗБИВКА ПО СМЫСЛОВЫМ ТИПАМ ==========
# 1. Числовые
numerical = matches.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 2. Категориальные (строки)
categorical = matches.select_dtypes(include=['object']).columns.tolist()

# 3. Дата/время
datetime_cols = matches.select_dtypes(include=['datetime64']).columns.tolist()
# Если даты не распознаны автоматически — ищем вручную
if not datetime_cols:
    datetime_cols = [c for c in matches.columns if 'date' in c.lower()]

# 4. Бинарные (2 уникальных значения)
binary_cols = []
for col in matches.columns:
    if col in datetime_cols:
        continue
    n_unique = matches[col].nunique(dropna=True)
    if n_unique == 2:
        binary_cols.append(col)

# 5. ID-колонки (просто идентификаторы)
id_cols = [c for c in matches.columns if 'id' in c.lower()]

# 6. Прочие (не попали ни в одну категорию)
known = set(numerical + categorical + datetime_cols + binary_cols + id_cols)
other_cols = [c for c in matches.columns if c not in known]

# ========== ВЫВОД ==========
print("="*60)
print("ТИПЫ ПРИЗНАКОВ")
print("="*60)
print(f"Числовые:        {len(numerical)}")
print(f"Категориальные:  {len(categorical)}")
print(f"Дата/время:      {len(datetime_cols)}  → {datetime_cols}")
print(f"Бинарные:        {len(binary_cols)}")
print(f"ID-колонки:      {len(id_cols)}")
print(f"Прочие:          {len(other_cols)}  → {other_cols}")

print(f"\nВсего колонок: {matches.shape[1]}")

ВСЕ ТИПЫ ДАННЫХ В ДАТАСЕТЕ (dtypes):
float64    96
object     11
int64       9
Name: count, dtype: int64

ТИПЫ ПРИЗНАКОВ
Числовые:        105
Категориальные:  11
Дата/время:      1  → ['date']
Бинарные:        5
ID-колонки:      6
Прочие:          0  → []

Всего колонок: 116


In [18]:
# Количество числовых признаков
numeric_cols = matches.select_dtypes(include=['int64', 'float64']).columns
n_numeric = len(numeric_cols)

print(f"Количество числовых признаков: {n_numeric}")

Количество числовых признаков: 105


In [19]:
# Количество категориальных признаков
categorical_cols = matches.select_dtypes(include=['object']).columns
n_categorical = len(categorical_cols)

print(f"Количество категориальных признаков: {n_categorical}")
print(f"\nСписок:")
print(categorical_cols.tolist())

Количество категориальных признаков: 11

Список:
['season', 'date', 'goal', 'shoton', 'shotoff', 'foulcommit', 'card', 'cross', 'corner', 'possession', 'result']
